# LensForge Common Test I

This notebook covers the required Common Test I multi-class classification task.

The selected retained result in LensForge is now the wider stratified `90:10` polar-view CNN run trained on the full combined train+val pool, reaching validation accuracy `0.9141` and validation macro ROC-AUC `0.9849`.
The polar representation is kept because it better matches the circular and arc-like morphology in the lensing images than the earlier small image-space CNN baselines, and the wider backbone closes much of the earlier performance gap.

Classes:
- `no`
- `sphere`
- `vort`


In [ ]:
from pathlib import Path
import json
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_ROOT = Path("data/common-test-i")
REPORT_PATH = Path("reports/common_test_i_polar_9010_noaug_w32.json")
MODEL_PATH = Path("models/common_test_i_polar_9010_noaug_w32.pt")


In [ ]:
rows = []
for split in ["train", "val"]:
    for cls in ["no", "sphere", "vort"]:
        files = sorted((DATA_ROOT / split / cls).glob("*.npy"))
        sample = np.load(files[0])
        rows.append({
            "split": split,
            "class": cls,
            "count": len(files),
            "shape": tuple(sample.shape),
            "dtype": str(sample.dtype),
            "min": float(sample.min()),
            "max": float(sample.max()),
        })
pd.DataFrame(rows)


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for row, cls in enumerate(["no", "sphere", "vort"]):
    sample = np.load(sorted((DATA_ROOT / "train" / cls).glob("*.npy"))[0])[0]
    for col in range(3):
        axes[row, col].imshow(sample, cmap="magma")
        axes[row, col].set_title(f"{cls} | example")
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
command = [
    "python3",
    "train_common_test_i.py",
    "--data-root",
    str(DATA_ROOT),
    "--epochs",
    "10",
    "--batch-size",
    "32",
    "--combine-train-and-val",
    "--validation-fraction",
    "0.1",
    "--model-type",
    "polar",
    "--view-mode",
    "polar",
    "--polar-radius",
    "72",
    "--polar-height",
    "80",
    "--polar-width",
    "96",
    "--width",
    "32",
    "--normalize-mode",
    "per_image_standardize",
    "--label-smoothing",
    "0.03",
    "--disable-augment",
    "--report-path",
    str(REPORT_PATH),
    "--model-path",
    str(MODEL_PATH),
]
completed = subprocess.run(command, check=True, text=True, capture_output=True)
print(completed.stdout)


In [ ]:
report = json.loads(REPORT_PATH.read_text())
pd.DataFrame([report["best_validation"]])


In [ ]:
history = pd.DataFrame([
    {
        "epoch": row["epoch"],
        "train_loss": row["train"]["loss"],
        "train_acc": row["train"]["accuracy"],
        "train_auc": row["train"]["macro_roc_auc"],
        "val_loss": row["validation"]["loss"],
        "val_acc": row["validation"]["accuracy"],
        "val_auc": row["validation"]["macro_roc_auc"],
    }
    for row in report["history"]
])
history


## Experiment log

Several Common Test I variants have now been tested. The current best finished result in this repository is the wider stratified 90:10 polar-view CNN without augmentation, trained on the full combined train+val pool. Its retained best checkpoint reaches validation accuracy 0.9141 and validation macro ROC-AUC 0.9849, substantially improving on the earlier retained polar runs and showing that geometry-aware representations plus a stronger polar backbone are the most promising direction found in LensForge for this task.


In [ ]:
experiments = pd.DataFrame(json.loads(Path("reports/common_test_i_experiments.json").read_text()))
experiments


In [ ]:
compact = pd.DataFrame(json.loads(Path("reports/common_test_i_experiments_compact.json").read_text()))
compact.sort_values(["val_macro_roc_auc", "val_accuracy"], ascending=False)
